# Análise exploratória — VPA e P/VPA

Pré-requisito: `python -m vpa_b3 --so-vpa` (gera `data/processed/vpa_trimestral.parquet`).
Para os resultados do backtest, rode `python -m vpa_b3` antes.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
from vpa_b3 import config, precos, processamento, estrategia
from vpa_b3.config import ParamsEstrategia

vpa = pd.read_parquet(config.DIR_PROC / 'vpa_trimestral.parquet')
vpa.groupby('ticker').agg(trimestres=('ref_date', 'size'), inicio=('ref_date', 'min'),
                          pct_consolidado=('statement_type', lambda s: (s == 'consolidado').mean()),
                          outliers=('flag_outlier', 'sum'), pl_negativo=('flag_pl_negativo', 'sum')).sort_values('inicio')

## Defasagem de divulgação
`receipt_date − ref_date` por tipo de documento; a regra `avail_date = max(receipt, ref + 3 meses)` só adia ITRs entregues antes de 90 dias.

In [ ]:
lag = (vpa['receipt_date'] - vpa['ref_date']).dt.days
lag.groupby(vpa['fonte'].str.extract(r'(DFP|ITR)')[0]).describe()

## P/VPA de um ativo com as bandas da estratégia
Média expandida e média + 2σ (até t−1, por regime).

In [ ]:
TICKER = 'PETR4'
px = precos.baixar_ativo(TICKER)
est = estrategia.estatisticas(processamento.pvpa_diario(vpa[vpa.ticker == TICKER], px), 8)
sin = estrategia.sinais(est, ParamsEstrategia())
fig, ax = plt.subplots(figsize=(11, 4))
est['pvpa'].plot(ax=ax, lw=1, label='P/VPA')
est['media'].plot(ax=ax, lw=1.5, label='média expandida')
(est['media'] + 2 * est['desvio']).plot(ax=ax, lw=1, ls='--', label='média + 2σ')
ax.scatter(est.index[sin['compra'] & ~sin['compra'].shift(fill_value=False)], est['pvpa'][sin['compra'] & ~sin['compra'].shift(fill_value=False)], s=12, label='entra em zona de compra')
ax.set_title(TICKER); ax.legend(); plt.show()

## Auditoria
Eventos registrados no processamento (splits não registrados, quebras estruturais, outliers).

In [ ]:
aud = pd.read_csv(config.DIR_REPORTS / 'auditoria_vpa.csv')
aud.pivot_table(index='ticker', columns='evento', aggfunc='size', fill_value=0)

## Resultado por ativo

In [ ]:
pa = pd.read_csv(config.DIR_REPORTS / 'metricas_por_ativo.csv')
pa[['ticker', 'cagr', 'bh_cagr', 'dif_cagr', 'dif_cagr_ic_lo', 'dif_cagr_ic_hi', 'pct_tempo_exposto', 'n_trades', 'nunca_vendeu']].round(3)